In [4]:
from langchain_ollama import ChatOllama
import requests

session = requests.Session()
session.trust_env = False

r = session.get("http://127.0.0.1:11434/api/tags")
print(r.text)

llm = ChatOllama(
    model="deepseek-r1:7b",
    temperature=0,
    base_url = "http://127.0.0.1:11434",
    client_kwargs={"trust_env":False},
)

resp = llm.invoke("用一句话解释什么是TCP")
print(resp.content)

{"models":[{"name":"qwen3-embedding:0.6b","model":"qwen3-embedding:0.6b","modified_at":"2026-09-11T14:14:15.4704928+08:00","size":639150858,"digest":"ac6da0dfba84a81fdbfbaf330198c33cd77c4cdfc53e8bc50eb581914a15621d","details":{"parent_model":"/Users/ollama/.ollama/models/blobs/sha256-06507c7b42688469c4e7298b0a1e16deff06caf291cf0a5b278c308249c3e439","format":"gguf","family":"qwen3","families":["qwen3"],"parameter_size":"595.78M","quantization_level":"Q8_0","context_length":32768,"embedding_length":1024},"capabilities":["embedding"]},{"name":"deepseek-r1:7b","model":"deepseek-r1:7b","modified_at":"2026-09-10T19:59:51.6303127+08:00","size":4683075440,"digest":"755ced02ce7befdb13b7ca74e1e4d08cddba4986afdb63a480f2c93d3140383f","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"7.6B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":3584},"capabilities":["completion","thinking"]}]}


TCP（传输控制协议）是一个可靠、面向连接的协议，用于建立连接、分段重组

In [11]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content="""
TCP 是一种面向连接、可靠的传输层协议。
TCP 通过序号、确认、重传等机制实现可靠传输。
"""
    ),
    Document(
        page_content="""
TCP 建立连接通常使用三次握手。
第一次客户端发送 SYN，
第二次服务器返回 SYN+ACK，
第三次客户端发送 ACK。
"""
    ),
    Document(
        page_content="""
UDP 是一种无连接的传输层协议。
UDP 不保证可靠交付，也不提供重传和流量控制。
"""
    ),
    Document(
        page_content="""
ARP 用于根据 IPv4 地址查询同一局域网中的 MAC 地址。
ARP 工作在局域网环境中。
"""
    ),
]

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5"
)

d:\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\25175\.cache\huggingface\hub\models--BAAI--bge-small-zh-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.mi

In [14]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings
)

In [15]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

In [22]:
results = retriever.invoke("TCP为什么需要三次握手？")

for i, doc in enumerate(results, 1):
    print(f"--- 文档{i} ---")
    print(doc.page_content)

--- 文档1 ---

TCP 建立连接通常使用三次握手。
第一次客户端发送 SYN，
第二次服务器返回 SYN+ACK，
第三次客户端发送 ACK。

--- 文档2 ---

TCP 是一种面向连接、可靠的传输层协议。
TCP 通过序号、确认、重传等机制实现可靠传输。



In [17]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
你是一名计算机网络助教。

请严格根据下面给出的资料回答问题。
如果资料中没有答案，请明确说“根据当前知识库无法确定”。

资料：
{context}

问题：
{question}
""")

In [18]:
def ask(question):
    docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    response = llm.invoke(messages)

    return response.content

In [20]:
print(ask("UDP是啥"))



UDP（用户数据报协议）是一种无连接的传输层协议，主要用于实现快速的数据传输。它不保证数据的可靠传输，不提供重传机制或流量控制，因此在数据可能丢失的情况下，数据可能会丢失而不被重传。这种特性使得UDP适合那些对数据可靠性要求不高的应用场景，如实时视频传输或在线游戏，其中延迟和数据完整性比可靠性更为重要。


In [ ]:
def chunk_by_paragraphs(markdown,max_chars=300):
    